In [1]:
import polars as pl
from depersonal_lib.core.privacy_models import KAnonymity
from depersonal_lib.core.privacy_models import LDiversity
from depersonal_lib.core.privacy_models import TCloseness

## Пример k-анонимности

In [8]:
# Создаем тестовые данные
df = pl.DataFrame({
    "age": [25, 27, 30, 32, 45],
    "gender": ['M','M','F','F','M'],
    "zipcode": [10001, 10001, 10002, 10002, 10003]
})
df

age,gender,zipcode
i64,str,i64
25,"""M""",10001
27,"""M""",10001
30,"""F""",10002
32,"""F""",10002
45,"""M""",10003


In [9]:
# Анализ анонимности по квази-идентификаторам [age, zipcode]
kan = KAnonymity(["gender", "zipcode"])
min_k, weak = kan.analyze_k(df)
print('Текущий уровень k-анонимности: ',min_k)

Текущий уровень k-анонимности:  1


In [11]:
print("\nСамые маленькие группы:")
weak


Самые маленькие группы:


gender,zipcode,group_size
str,i64,u32
"""M""",10003,1


## Проверка l-разнообразия

In [12]:
# Создаем тестовые данные
df = pl.DataFrame({
    "gender": ['M','F','M','F','M'],
    "zipcode": [10001, 10001, 10001, 10002, 10002],
    "diagnosis": ['Flu', 'Flu','Cold','Allergy','Cold']
})
df

gender,zipcode,diagnosis
str,i64,str
"""M""",10001,"""Flu"""
"""F""",10001,"""Flu"""
"""M""",10001,"""Cold"""
"""F""",10002,"""Allergy"""
"""M""",10002,"""Cold"""


In [20]:
# Анализ l-разнообразия
ldiv = LDiversity(["gender", "zipcode"], "diagnosis")
min_l, weak_groups = ldiv.analyze_l(df)
print(f"Текущее l-разнообразие: {min_l}")

Текущее l-разнообразие: 1


In [21]:
weak_groups

gender,zipcode,unique_count
str,i64,u32
"""M""",10002,1
"""F""",10001,1
"""F""",10002,1


## Проверка t-близости

In [2]:
# Создаем тестовые данные
df = pl.DataFrame({
    "gender": ['M','F','M','F','M','F'],
    "zipcode": [10001, 10001, 10001, 10002, 10002, 10002],
    "condition": ['A', 'A','B','A','C','C']
})
df

gender,zipcode,condition
str,i64,str
"""M""",10001,"""A"""
"""F""",10001,"""A"""
"""M""",10001,"""B"""
"""F""",10002,"""A"""
"""M""",10002,"""C"""
"""F""",10002,"""C"""


In [3]:
# Анализ t-близости
t_close = TCloseness(["gender", "zipcode"], "condition")

In [4]:
max_t, weak_groups = t_close.analyze_t(df)

In [5]:
print(f"Максимальное t-расстояыние: {max_t:.4f}")
print("Группы с максимальным отклонением:")
print(weak_groups)

Максимальное t-расстояыние: 0.6667
Группы с максимальным отклонением:
shape: (1, 3)
┌────────┬─────────┬────────────┐
│ gender ┆ zipcode ┆ t_distance │
│ ---    ┆ ---     ┆ ---        │
│ str    ┆ i64     ┆ f64        │
╞════════╪═════════╪════════════╡
│ M      ┆ 10002   ┆ 0.666667   │
└────────┴─────────┴────────────┘


## 